In [0]:
# ---------- Cell 1: settings ----------
from pyspark.sql import functions as F
from pyspark.sql.window import Window
CATALOG_SCHEMA = "workspace.stocks"

In [0]:
typed = spark.table(f"{CATALOG_SCHEMA}.bronze_prices")

casts = {"trade_date": "DATE",
    "open":  "DECIMAL(18,4)", "high": "DECIMAL(18,4)",
    "low":   "DECIMAL(18,4)", "close": "DECIMAL(18,4)",
    "volume": "BIGINT",}

for col_name, sql_type in casts.items():
    typed = (typed.withColumnRenamed(col_name,f"{col_name}_raw")
                     .withColumn(col_name,F.expr(f"try_cast({col_name}_raw AS {sql_type})")))

In [0]:
bad_condition = (
    (F.col("trade_date").isNull() & F.col("trade_date_raw").isNotNull()) |
    (F.col("close").isNull()      & F.col("close_raw").isNotNull()) |
    (F.col("open").isNull()      & F.col("open_raw").isNotNull())|
    (F.col("low").isNull()      & F.col("low_raw").isNotNull()) |
    (F.col("high").isNull()      & F.col("high_raw").isNotNull()) |
    (F.col("volume").isNull()      & F.col("volume_raw").isNotNull() | F.col("_rescued_data").isNotNull())

)
bad  = typed.filter(bad_condition)
good = typed.filter(~bad_condition)

w = Window.partitionBy("ticker","trade_date").orderBy(F.col("load_ts").desc(),F.col("source_file").desc())
silver_good = good.withColumn("rn",F.row_number().over(w)).filter(F.col("rn")==1)

raw_cols = [f"{c}_raw" for c in casts]         
silver_final = silver_good.drop("rn","_rescued_data", *raw_cols)

In [0]:
silver_final.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG_SCHEMA}.silver_prices")
bad.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG_SCHEMA}.silver_prices_rejects")

In [0]:
sp = spark.table(f"{CATALOG_SCHEMA}.silver_prices")
print("silver rows:", sp.count())
print("unique (ticker, trade_date):", sp.select("ticker", "trade_date").distinct().count())
print("rejects:", spark.table(f"{CATALOG_SCHEMA}.silver_prices_rejects").count())
sp.printSchema()

silver rows: 800
unique (ticker, trade_date): 800
rejects: 0
root
 |-- ticker: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- load_ts: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- trade_date: date (nullable = true)
 |-- open: decimal(18,4) (nullable = true)
 |-- high: decimal(18,4) (nullable = true)
 |-- low: decimal(18,4) (nullable = true)
 |-- close: decimal(18,4) (nullable = true)
 |-- volume: long (nullable = true)

